This notebook accomplishes two things:
- process divide attributes for all NWM domains (except for 'gl' where divide-attributes layer is not available), removing inconsistencies
- contruct the initital attributes configuration file (that can later be used to select attributes for regionalization)

In [1]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

In [ ]:
# define data directories
gpkg_dir = Path("/home/yuqiong.liu/work/data/gpkg_v2.2/")
ngen_reg_dir = Path("/home/yuqiong.liu/work/data/ngen_reg/")

In [ ]:
# loop through all NWM domains to process and reconcile divide attributes (except for 'gl' where divide-attributes layer is not available)
domains = ["conus", "ak", "hi", "prvi"]
attr_list = dict()
for domain in domains:
    print(domain)

    # read NextGen hydrofabric divide-attributes
    gpkg_ngen = Path(gpkg_dir, domain + "_nextgen.gpkg").resolve()
    df_attr = gpd.read_file(gpkg_ngen, layer="divide-attributes")
    attr_list[domain] = df_attr.columns.to_list()
    print(f"Number of columns: {len(df_attr.columns)}")

    # aggregate soil parameters associated with multiple soil layers
    soil_cols = [
        c1 for c1 in df_attr.columns if c1 == "divide_id" or "soil_layers_stag" in c1
    ]
    df_soil = df_attr[soil_cols]
    df_soil_long = df_soil.melt(
        id_vars=["divide_id"], var_name="group", value_name="value"
    )
    df_soil_long["group"] = df_soil_long["group"].str.replace(
        "_soil_layers_stag=1|_soil_layers_stag=2|"
        "_soil_layers_stag=3|_soil_layers_stag=4",
        "",
        regex=True,
    )
    df_soil_agg = (
        df_soil_long.groupby(["group", "divide_id"])["value"]
        .mean()
        .reset_index(name="result")
    )
    df_soil_agg = df_soil_agg.pivot(index="divide_id", columns="group", values="result")
    df_soil_agg.reset_index(inplace=True)

    # merge with the non-soil parameters
    non_soil_cols = [c1 for c1 in df_attr.columns if "soil_layers_stag" not in c1]
    df_non_soil = df_attr[non_soil_cols]
    df_attr_new = df_soil_agg.merge(df_non_soil, on="divide_id", how="outer")

    # remove 'mean.', 'geomean.', 'mode.' from column names
    df_attr_new.columns = df_attr_new.columns.str.replace(
        "mean.|geom_|mode.|circ_", "", regex=True
    )

    # for prvi, rename 'slope' to be'slope_1km' to be consistent with other domains
    # some prvi parameters have an extra '_Time=' in their names
    if domain == "prvi":
        df_attr_new = df_attr_new.rename(columns={"slope_Time=": "slope_1km"})
        df_attr_new.columns = df_attr_new.columns.str.replace("_Time=", "", regex=False)

    # for oCONUS domains, change X/Y to centroid_x/centrod_y to be consistent with CONUS
    if domain != "conus":
        df_attr_new = df_attr_new.rename(columns={"X": "centroid_x", "Y": "centroid_y"})

    # write to file for the domain
    # outfile = Path(ngen_reg_dir, 'attr_ngen/attr_ngen_' + domain + '.csv')
    outfile = Path(ngen_reg_dir, "attr_datasets/attr_ngen_" + domain + ".parquet")
    if not outfile.parent.exists():
        outfile.parent.mkdir(parents=True, exist_ok=True)
    # df_attr_new.to_csv(outfile, header=True, index=False)
    df_attr_new.to_parquet(outfile)

    # also write to by individual VPUs
    vpus = df_attr_new["vpuid"].unique()
    if len(vpus) > 1:
        for vpu in vpus:
            df1 = df_attr_new[df_attr_new["vpuid"] == vpu]
            outfile1 = Path(str(outfile).replace(domain, domain + "_" + vpu))
            df1.to_parquet(outfile1)

# save the original column names to file
attr_file = Path(ngen_reg_dir, "nextgen_gpkg_columns.csv")
pd.DataFrame(attr_list).to_csv(attr_file, index=False, header=True)

In [ ]:
# loop through the domains to gather attributes (that are numeric usable by regionalization tool)
domains = ["conus", "ak", "hi", "prvi"]
all_attrs = []
for domain in domains:
    # print(domain)
    # file1 = Path(ngen_reg_dir, 'attr_ngen/attr_ngen_' + domain + '.csv')
    file1 = Path(ngen_reg_dir, "attr_datasets/attr_ngen_" + domain + ".parquet")
    df_attr = pd.read_parquet(file1)
    # df_attr = pd.read_csv(file1,low_memory=False)
    # print(df_attr[['ISLTYP','IVGTYP', 'impervious', 'Coeff']].describe())
    numeric_columns = df_attr.select_dtypes(include="number").columns.to_list()
    all_attrs = all_attrs + [x for x in numeric_columns if x not in all_attrs]

print(all_attrs)

['dksat', 'psisat', 'smcmax', 'smcwlt', 'bexp', 'ISLTYP', 'IVGTYP', 'cwpvt', 'mfsno', 'mp', 'refkdt', 'slope_1km', 'vcmx25', 'Coeff', 'Zmax', 'Expon', 'centroid_x', 'centroid_y', 'impervious', 'elevation', 'slope', 'aspect']


In [8]:
# Prepare the attributes config file (based on all_attrs gathered above)
df_attr_config = pd.DataFrame(
    columns=["select", "attr_name", "description"],
    data=[
        (1, "dksat", "NWM parameter | saturated hydraulic conductivity"),
        (1, "psisat", "NWM parameter | saturated capillary head"),
        (1, "smcmax", "NWM parameter | saturated soil moisture content"),
        (1, "smcwlt", "NWM parameter | wilting point soil moisture content"),
        (
            1,
            "bexp",
            "NWM parameter | beta exponent on Clapp-Hornberger (1978) soil water relations",
        ),
        (
            1,
            "cwpvt",
            "NWM parameter | Canopy wind parameter for canopy wind profile formulation",
        ),
        (1, "mfsno", "NWM parameter | Melt factor for snow depletion curve"),
        (1, "mp", "NWM parameter | Slope of Ball-Berry conductance relationship"),
        (1, "refkdt", "NWM parameter | Soil infiltration parameter"),
        (
            1,
            "slope_1km",
            "NWM parameter | Coeffecient controlling the drainage out of the soil bottom (0=no-flow)",
        ),
        (1, "vcmx25", "NWM parameter | Maximum carboxylation at 25 degC"),
        # (1, 'Coeff', 'NWM parameter | constant(0.005)'),
        (1, "Zmax", "NWM parameter | maximum storage in the conceptual reservoir"),
        (
            1,
            "Expon",
            "NWM parameter | exponent for nonlinear ground water reservoir (1.0 for linear reservoir)",
        ),
        (1, "centroid_x", "centroid_x of catchment"),
        (1, "centroid_y", "centroid_y of catchment"),
        (0, "impervious", "percentage of impervious area"),
        (1, "elevation", "mean elevation catchment"),
        (1, "slope", "mean slope of catchment"),
        (1, "aspect", "mean aspect of catchment"),
        (0, "ISLTYP", "dominant soil type"),
        (0, "IVGTYP", "dominant vegetation class"),
    ],
)

df_attr_config.to_csv(
    Path(ngen_reg_dir, "config/attr_selection_ngen.csv"), index=False, header=True
)

In [ ]:
print(df_attr_config)